## Section 4.2: analyst-guided projection grid search

This notebook contains the grid-search experiment used to produce Table 2 in the paper. We evaluate four AES trace settings: hardware AES, hardware AES after low-pass filtering, software AES, and software AES after low-pass filtering.

The shared implementation in spectroloc.self_temp_analyst handles trace loading and the basic evaluation for one projection method, window size, and percentile setting. The notebook specifies the experiment configuration, including dataset paths, trigger thresholds, and the remaining grid-search settings.

Define the datasets and grid-search parameters used for Table 2.

In [1]:
from spectroloc.config import AnalystDatasetConfig
from spectroloc.self_temp_analyst import load_data, run_single_combination
import numpy as np
import random
import time

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

WINDOW_LIST = [1000, 5000, 10000, 15000, 20000, 25000, 30000]
PERCENTILE_LIST = [0.1, 1, 2, 4, 6, 8, 10]
METHOD_COMBINATIONS = [
    ("stft", "l1"),
    ("stft", "l2"),
    ("time", "l1"),
    ("time", "l2"),
]

trigger_threshold_raw = 190.0

configs = [
    AnalystDatasetConfig(
        name="HW AES",
        signal_path="./dataset/semi-loc/stm32f4/STM32F4_HWAES_500MSPS_50M_NoLowPass.npy",
        trigger_path="./dataset/semi-loc/stm32f4/STM32F4_HWAES_500MSPS_50M_NoLowPass_trig.npy",
        trigger_threshold_raw=trigger_threshold_raw,
    ),
    AnalystDatasetConfig(
        name="HW AES (Low-pass)",
        signal_path="./dataset/semi-loc/stm32f4/STM32F4_HWAES_500MSPS_50M_NoLowPass_LowPass_Ratio0.05.npy",
        trigger_path="./dataset/semi-loc/stm32f4/STM32F4_HWAES_500MSPS_50M_NoLowPass_trig.npy",
        trigger_threshold_raw=trigger_threshold_raw,
    ),
    AnalystDatasetConfig(
        name="SW AES",
        signal_path="./dataset/semi-loc/stm32f4/STM32F4_TinyAes_125MSPS_50M_NoLowPass_new.npy",
        trigger_path="./dataset/semi-loc/stm32f4/STM32F4_TinyAes_125MSPS_50M_NoLowPass_new_trig.npy",
        trigger_threshold_raw=trigger_threshold_raw,
    ),
    AnalystDatasetConfig(
        name="SW AES (Low-pass)",
        signal_path="./dataset/semi-loc/stm32f4/STM32F4_TinyAes_125MSPS_50M_NoLowPass_new_LowPass_Ratio0.05.npy",
        trigger_path="./dataset/semi-loc/stm32f4/STM32F4_TinyAes_125MSPS_50M_NoLowPass_new_trig.npy",
        trigger_threshold_raw=trigger_threshold_raw,
    ),
]


Define the helpers for running the grid search and printing the Table 2-style summary. The paper-specific details, including dataset order, grid layout, and output formatting, are handled here.

In [2]:
def print_grid_search_summary(results, method_combinations):
    """Print the Table 2 grid-search hit rates in a notebook-local format."""

    if not results:
        print("\nNo grid-search results to summarize.")
        return

    datasets = list(dict.fromkeys(r.dataset_name for r in results))
    rows = []
    for method, agg in method_combinations:
        row = {"Method": f"{method}+{agg}"}
        for dataset in datasets:
            result = next(
                (
                    r
                    for r in results
                    if r.method == method and r.agg == agg and r.dataset_name == dataset
                ),
                None,
            )
            row[dataset] = f"{result.accuracy:.2f}%" if result else "N/A"
        rows.append(row)

    columns = ["Method", *datasets]
    column_width = max(
        max(len(column) for column in columns),
        max(len(str(row[column])) for row in rows for column in columns),
    ) + 2

    print("\n" + "=" * 20 + " Mean Hit Rate " + "=" * 20)
    print(" " * column_width + "".join(dataset.center(column_width) for dataset in datasets).rstrip())
    print("Method".ljust(column_width).rstrip())
    for row in rows:
        values = [row["Method"].ljust(column_width)]
        values.extend(row[dataset].center(column_width) for dataset in datasets)
        print("".join(values).rstrip())


def run_grid_search_experiment(configs, window_list, percentile_list, method_combinations):
    """Notebook-local orchestration for the paper's analyst-guided sweep."""

    summary_list = []
    for config in configs:
        signal_data, trigger_data = load_data(config)
        if signal_data is None:
            continue

        total = len(method_combinations) * len(window_list) * len(percentile_list)
        print(f"Starting experiments on {config.name} (Total combinations: {total})")

        for method, agg in method_combinations:
            t0 = time.time()
            result = run_single_combination(
                signal_data,
                trigger_data,
                config,
                method,
                agg,
                window_list,
                percentile_list,
            )
            summary_list.append(result)
            elapsed = time.time() - t0
            print(f"     Finished {method}-{agg} | Acc: {result.accuracy:.2f}% | Time: {elapsed:.2f}s\n")

    print_grid_search_summary(summary_list, method_combinations)
    return summary_list


Run the complete analyst-guided grid search and print the mean hit-rate table.

In [3]:
grid_search_results = run_grid_search_experiment(configs, WINDOW_LIST, PERCENTILE_LIST, METHOD_COMBINATIONS)



Loading data for dataset: HW AES...
Starting experiments on HW AES (Total combinations: 196)
  -> Testing combination: Method=stft, Agg=l1
     Finished stft-l1 | Acc: 14.29% | Time: 3.35s

  -> Testing combination: Method=stft, Agg=l2
     Finished stft-l2 | Acc: 0.00% | Time: 2.84s

  -> Testing combination: Method=time, Agg=l1
     Finished time-l1 | Acc: 0.00% | Time: 1.60s

  -> Testing combination: Method=time, Agg=l2
     Finished time-l2 | Acc: 0.00% | Time: 1.22s


Loading data for dataset: HW AES (Low-pass)...
Starting experiments on HW AES (Low-pass) (Total combinations: 196)
  -> Testing combination: Method=stft, Agg=l1
     Finished stft-l1 | Acc: 20.41% | Time: 2.98s

  -> Testing combination: Method=stft, Agg=l2
     Finished stft-l2 | Acc: 0.00% | Time: 2.74s

  -> Testing combination: Method=time, Agg=l1
     Finished time-l1 | Acc: 0.00% | Time: 1.22s

  -> Testing combination: Method=time, Agg=l2
     Finished time-l2 | Acc: 0.00% | Time: 1.27s


Loading data for da